In [1]:
!pip install ccxt

In [2]:
# ============================================
# 做市商：逆向选择 (Adverse Selection)
# ============================================

# ---------- Part 1: P&L Attribution ----------
pnl = {
    'spread_pnl': 120.50,
    'adverse_selection_pnl': -85.30,
    'inventory_pnl': 10.20
}
pnl['total_pnl'] = pnl['spread_pnl'] + pnl['adverse_selection_pnl'] + pnl['inventory_pnl']
adverse_ratio = abs(pnl['adverse_selection_pnl']) / pnl['spread_pnl'] * 100

print("=== P&L Attribution ===")
for k, v in pnl.items():
    print(f"  {k}: {v}")
print(f"  逆向选择吞噬价差比例: {adverse_ratio:.1f}%")
print()

# ---------- Part 2: Toxic Flow Detection ----------
def is_toxic_order(order, avg_order_size, recent_price, volatility, avg_volatility):
    is_large_size = order['size'] > avg_order_size * 3
    is_aggressive = (
        (order['side'] == 'sell' and order['price'] < recent_price * 0.99) or
        (order['side'] == 'buy' and order['price'] > recent_price * 1.01)
    )
    is_high_vol = volatility > avg_volatility * 1.5
    return is_large_size or (is_aggressive and is_high_vol)

print("=== Toxic Flow Detection ===")
test_orders = [
    {'side': 'sell', 'size': 5.0, 'price': 1980},
    {'side': 'buy', 'size': 1.0, 'price': 2000},
    {'side': 'sell', 'size': 2.5, 'price': 2005}
]
for i, o in enumerate(test_orders):
    result = is_toxic_order(o, avg_order_size=1.0, recent_price=2000,
                            volatility=25, avg_volatility=15)
    print(f"  Order {i+1}: {o} -> Toxic: {result}")
print()

# ---------- Part 3: Dynamic Spread Adjustment ----------
def adjusted_spread(base_spread, adverse_selection_risk):
    return base_spread * (1 + adverse_selection_risk)

print("=== Dynamic Spread Adjustment ===")
for risk in [0, 0.3, 0.5, 0.8, 1.0]:
    result = adjusted_spread(0.10, risk)
    print(f"  Base Spread: 0.10%, Risk: {risk} -> Adjusted: {result:.3f}%")
print()

# ---------- Part 4: Combine it all ----------
print("=== Full Workflow Demo ===")
live_spread = 0.12
live_orders = [
    {'side': 'buy', 'size': 0.5, 'price': 2001, 'label': 'small buy'},
    {'side': 'sell', 'size': 8.0, 'price': 1985, 'label': 'large sell (suspicious)'},
    {'side': 'buy', 'size': 1.2, 'price': 2020, 'label': 'aggressive buy (suspicious)'}
]
for o in live_orders:
    toxic = is_toxic_order(o, avg_order_size=1.0, recent_price=2000,
                           volatility=25, avg_volatility=15)
    risk = 0.5 if toxic else 0.1
    adj = adjusted_spread(live_spread, risk)
    print(f"  {o['label']}: Toxic={toxic}, Risk={risk}, Adjusted Spread={adj:.3f}%")

=== P&L Attribution ===
  spread_pnl: 120.5
  adverse_selection_pnl: -85.3
  inventory_pnl: 10.2
  total_pnl: 45.400000000000006
  逆向选择吞噬价差比例: 70.8%

=== Toxic Flow Detection ===
  Order 1: {'side': 'sell', 'size': 5.0, 'price': 1980} -> Toxic: True
  Order 2: {'side': 'buy', 'size': 1.0, 'price': 2000} -> Toxic: False
  Order 3: {'side': 'sell', 'size': 2.5, 'price': 2005} -> Toxic: False

=== Dynamic Spread Adjustment ===
  Base Spread: 0.10%, Risk: 0 -> Adjusted: 0.100%
  Base Spread: 0.10%, Risk: 0.3 -> Adjusted: 0.130%
  Base Spread: 0.10%, Risk: 0.5 -> Adjusted: 0.150%
  Base Spread: 0.10%, Risk: 0.8 -> Adjusted: 0.180%
  Base Spread: 0.10%, Risk: 1.0 -> Adjusted: 0.200%

=== Full Workflow Demo ===
  small buy: Toxic=False, Risk=0.1, Adjusted Spread=0.132%
  large sell (suspicious): Toxic=True, Risk=0.5, Adjusted Spread=0.180%
  aggressive buy (suspicious): Toxic=False, Risk=0.1, Adjusted Spread=0.132%
